# Mushroom Edibility Classification

**Tabular Classification Project**

## 2 · Project Overview

Classify mushrooms as **edible or poisonous** from 10 physical attributes: cap shape, cap color, odor, gill size, stalk shape, ring type, spore print color, population, habitat, and bruises. Synthetic dataset with ~8,000 rows and ~48% edible rate. This is a classic nearly-perfectly-separable dataset where **odor** alone achieves ~98% accuracy.

## 3 · Learning Objectives

1. Perform EDA and target analysis on a classification dataset.
2. Handle categorical encoding, missing values, and class imbalance.
3. Build a baseline model and compare against modern boosting models.
4. Use LazyPredict for rapid classifier benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, and XGBoost with GPU auto-detection.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given 10 categorical attributes describing a mushroom's physical appearance, classify it as edible (1) or poisonous (0).

## 5 · Why This Project Matters

- **Mushroom foraging safety** — misclassification can be fatal.
- This dataset is famous for near-perfect separability from a single feature (odor).
- Teaches the importance of **checking simple rules before complex models**.
- Safety-critical ML requires very high recall on the dangerous class.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| Rows | ~8,000 |
| Features | 10 (cap_shape, cap_color, odor, gill_size, stalk_shape, ring_type, spore_print_color, population, habitat, bruises) |
| Target | `edible` (binary: 0=poisonous, 1=edible) |
| Class balance | ~52% poisonous, ~48% edible |
| Missing values | None |

## 7 · Dataset Source and License Notes

**Source**: UCI Mushroom dataset (inspired by Audubon Field Guide).
**License**: Public domain / educational.
**Note**: Synthetic reproduction of mushroom attribute patterns.

## 8 · Environment Setup

In [ ]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)
print("All packages ready.")

## 9 · Imports

In [ ]:
import os, json, time, warnings, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

## 10 · Configuration / Constants

In [ ]:
TARGET = "edible"
SEED = 42
SAVE_DIR = os.path.dirname(os.path.abspath('__dummy__'))
print(f"Save dir: {SAVE_DIR}")

## 11 · Dataset Download or Loading

In [ ]:
np.random.seed(SEED)
n = 8124

cap_shape = np.random.choice(['bell', 'conical', 'convex', 'flat', 'knobbed', 'sunken'], n)
cap_color = np.random.choice(['brown', 'buff', 'cinnamon', 'gray', 'green', 'pink', 'purple', 'red', 'white', 'yellow'], n)
odor = np.random.choice(['almond', 'anise', 'creosote', 'fishy', 'foul', 'musty', 'none', 'pungent', 'spicy'], n,
                         p=[0.05, 0.05, 0.05, 0.1, 0.2, 0.05, 0.35, 0.1, 0.05])
gill_size = np.random.choice(['broad', 'narrow'], n, p=[0.6, 0.4])
stalk_shape = np.random.choice(['enlarging', 'tapering'], n)
ring_type = np.random.choice(['evanescent', 'flaring', 'large', 'none', 'pendant'], n)
spore_color = np.random.choice(['black', 'brown', 'buff', 'chocolate', 'green', 'orange', 'purple', 'white', 'yellow'], n)
population = np.random.choice(['abundant', 'clustered', 'numerous', 'scattered', 'several', 'solitary'], n)
habitat = np.random.choice(['grasses', 'leaves', 'meadows', 'paths', 'urban', 'waste', 'woods'], n)
bruises = np.random.choice(['yes', 'no'], n, p=[0.4, 0.6])

# Odor is the dominant predictor (matching original dataset behavior)
edible_odors = {'almond', 'anise', 'none'}
score = np.array([1.0 if o in edible_odors else -1.0 for o in odor])
score += 0.3 * (gill_size == 'broad').astype(float)
score += 0.2 * (bruises == 'yes').astype(float)
score -= 0.2 * (spore_color == 'green').astype(float)
score += np.random.normal(0, 0.15, n)
edible = (score > 0).astype(int)

df = pd.DataFrame({
    'cap_shape': cap_shape, 'cap_color': cap_color, 'odor': odor,
    'gill_size': gill_size, 'stalk_shape': stalk_shape, 'ring_type': ring_type,
    'spore_print_color': spore_color, 'population': population,
    'habitat': habitat, 'bruises': bruises, 'edible': edible,
})
print(f"Dataset shape: {df.shape}")
print(f"Edible rate: {df['edible'].mean():.2%}")
df.head()

## 12 · Data Validation Checks

In [ ]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
if df.isnull().sum().sum() == 0:
    print("No missing values.")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")

## 13 · Exploratory Data Analysis

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for i, col in enumerate(['odor', 'gill_size', 'bruises', 'spore_print_color', 'cap_shape', 'habitat']):
    ax = axes[i // 3, i % 3]
    ct = pd.crosstab(df[col], df[TARGET], normalize='index')
    ct.plot(kind='bar', stacked=True, ax=ax, color=['coral', 'steelblue'])
    ax.set_title(f"Edibility by {col}")
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('eda_numeric.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Categorical features: {len(cat_cols)}")

## 14 · Target Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df[TARGET].value_counts().plot(kind='bar', ax=axes[0], color=['coral', 'steelblue'], edgecolor='black')
axes[0].set_title("Mushroom Edibility")
axes[0].set_xticklabels(['Poisonous (0)', 'Edible (1)'], rotation=0)
df[TARGET].value_counts(normalize=True).plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['coral', 'steelblue'])
axes[1].set_title("Class Proportions"); axes[1].set_ylabel('')
plt.tight_layout(); plt.show()
print(f"Class distribution:\n{df[TARGET].value_counts()}")

## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 split to preserve class distribution.

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    df[cat_cols] = oe.fit_transform(df[cat_cols])

X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train target dist: {y_train.value_counts().to_dict()}")

## 16 · Preprocessing Strategy

Categorical features encoded via OrdinalEncoder. Missing values handled before split. Tree-based models handle raw features without scaling.

## 17 · Feature Engineering

In [ ]:
clean = [c.replace('-', '_').replace(' ', '_').replace('.', '_') for c in X_train.columns]
X_train.columns = clean; X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

## 18 · Baseline Model

Logistic Regression as baseline.

In [ ]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)
n_classes = len(np.unique(y_train))
if n_classes == 2:
    y_prob_bl = baseline.predict_proba(X_test)[:, 1]
else:
    y_prob_bl = None

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {accuracy_score(y_test, y_pred_bl):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_bl, average='weighted'):.4f}")
if y_prob_bl is not None:
    print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_bl):.4f}")

## 19 · LazyPredict Benchmark

In [ ]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)
print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

## 20 · FLAML AutoML Run

In [ ]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(
    X_train, y_train,
    task="classification", time_budget=60, metric="f1",
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best: {flaml_automl.best_estimator}")
print(f"Accuracy : {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_flaml, average='weighted'):.4f}")

## 21 · CatBoost, LightGBM, XGBoost

GPU auto-detected with CPU fallback.

In [ ]:
import gc

def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}

# CatBoost
from catboost import CatBoostClassifier
try:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            task_type="GPU", devices="0", verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
except Exception:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
results["CatBoost"] = cb.predict(X_test).ravel()
print(f"CatBoost  Acc: {accuracy_score(y_test, results['CatBoost']):.4f}  F1: {f1_score(y_test, results['CatBoost'], average='weighted'):.4f}")
gpu_cleanup()

# LightGBM
import lightgbm as lgbm
try:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              device="gpu", verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
except Exception:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
results["LightGBM"] = lg.predict(X_test)
print(f"LightGBM  Acc: {accuracy_score(y_test, results['LightGBM']):.4f}  F1: {f1_score(y_test, results['LightGBM'], average='weighted'):.4f}")
gpu_cleanup()

# XGBoost
from xgboost import XGBClassifier
try:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="cuda", tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
except Exception:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
results["XGBoost"] = xgb_model.predict(X_test)
print(f"XGBoost   Acc: {accuracy_score(y_test, results['XGBoost']):.4f}  F1: {f1_score(y_test, results['XGBoost'], average='weighted'):.4f}")
gpu_cleanup()

results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

## 22 · Top 3 Model Selection

In [ ]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average='weighted'), 4),
        "Precision": round(precision_score(y_test, yp, average='weighted'), 4),
        "Recall": round(recall_score(y_test, yp, average='weighted'), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())
top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3: {top3_names}")

## 23 · Final Training and Evaluation of Top 3

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap='Blues')
    f1 = f1_score(y_test, yp, average='weighted')
    acc = accuracy_score(y_test, yp)
    axes[i].set_title(f"{name}\nAcc={acc:.4f} F1={f1:.4f}")
plt.suptitle("Top 3 — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig('top3_confusion.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nDetailed Classification Reports — Top 3:")
for name in top3_names:
    print(f"\n{'='*50}")
    print(f"  {name}")
    print('='*50)
    print(classification_report(y_test, results[name]))

## 24 · Error Analysis

In [ ]:
best_name = top3_names[0]
best_pred = results[best_name]
errors = y_test != best_pred
error_rate = errors.mean()
print(f"Best model: {best_name}")
print(f"Error rate: {error_rate:.4f} ({errors.sum()} / {len(y_test)})")
print(f"\nErrors by true class:")
error_df = pd.DataFrame({'true': y_test, 'pred': best_pred, 'error': errors})
print(error_df.groupby('true')['error'].agg(['sum', 'count', 'mean']).rename(
    columns={'sum': 'errors', 'count': 'total', 'mean': 'error_rate'}))

## 25 · Interpretation and Business Insight

- **Odor** is the single most powerful feature — 'foul', 'pungent', 'creosote' indicate poison.
- Edible mushrooms tend to have 'none', 'almond', or 'anise' odor.
- **Gill size** and **bruises** provide additional signal.
- Models achieve near-perfect accuracy — the dataset is almost rule-based.
- A simple rule on odor alone gets ~98% accuracy.

## 26 · Limitations

1. Near-perfect separability makes this too easy for modern ML.
2. Synthetic attributes — real mushroom identification is far more complex.
3. Binary only — real toxicity has degrees (mild to lethal).
4. Missing key features: spore microscopy, geographic region, season.
5. Cannot replace expert mycological identification.

## 27 · How to Improve This Project

1. Add image-based classification using photos.
2. Include geographic and seasonal features.
3. Multi-level toxicity (mild, moderate, severe, lethal).
4. Combine with a field guide rule-based system.
5. Test on out-of-distribution mushroom species.

## 28 · Production Considerations

- **Safety-critical**: false negatives (calling poisonous edible) can be fatal.
- Must optimize for recall on the poisonous class.
- ML should supplement, not replace, expert identification.
- Requires extensive validation on real-world samples.

## 29 · Common Mistakes

1. Not checking if a single feature (odor) solves the problem.
2. Using accuracy when recall on poisonous class matters most.
3. Over-engineering with complex models on a nearly-separable dataset.
4. Not framing this as a safety-critical application.
5. Deploying without expert mycological validation.

## 30 · Mini Challenge / Exercises

1. Build a model using only 'odor' — what accuracy do you get?
2. Find the minimum feature set for 99%+ accuracy.
3. Calculate recall on the poisonous class — is it 100%?
4. Create a simple rule-based classifier from the decision tree.

## 31 · Final Summary / Key Takeaways

1. Mushroom classification is nearly perfectly solved by the 'odor' feature.
2. This teaches checking simple features before building complex models.
3. Safety-critical applications require near-perfect recall on dangerous classes.
4. Real mushroom identification needs expert knowledge beyond ML.
5. Simple rules often beat complex models on well-structured problems.

## Save Metrics

In [ ]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average='weighted')), 4),
        "precision": round(float(precision_score(y_test, yp, average='weighted')), 4),
        "recall": round(float(recall_score(y_test, yp, average='weighted')), 4),
    }
with open('metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)
print("Metrics saved.")
print(json.dumps(metrics_out, indent=2))